小鼠-gene-ratio

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================
# 1. 图表全局设置 (主刊级风格)
# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
mpl.rcParams['axes.linewidth'] = 1.2

# 新增：正则匹配辅助函数，避免无边界字符串的错误匹配 (如 'camp' 错误匹配到 'hippocampus')
def match_keywords(text, keywords):
    for k in keywords:
        # \b 表示单词边界，re.escape 确保特殊字符安全解析
        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================
# 2. 从联网数据库获取基因客观分类 (核糖体优先 + 严格正则匹配版)
# ==========================================
def get_objective_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    
    # 获取包含 GO 注释的完整信息
    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='mouse')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()

            # 提取 GO 注释并拼接
            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            
            # 构建综合检索文本
            search_text = desc + " " + go_terms + " " + gene_type

            # --- 严格正则分类判断引擎 (优先级层级过滤) ---
            
            # 1. 过滤未定义基因与假基因
            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            
            # 2. 【最高优先级】：核糖体、RNA加工与翻译
            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                
            # 3. 细胞骨架与 ECM
            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                
            # 4. 免疫与防御
            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                
            # 5. 细胞周期与凋亡
            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                
            # 6. 蛋白折叠与降解
            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                
            # 7. 表观遗传与染色质
            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                
            # 8. 转录调控
            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                
            # 9. 发育与分化
            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                
            # 10. 囊泡运输与细胞膜
            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                
            # 11. 信号传导与受体
            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                
            # 12. 代谢与转运
            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                
            # 13. 其他
            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================
# 3. 读取本地文件
# ==========================================
folder_path = "/mnt/e/2-8.3-shanda/1-feature/9-Final_Segmented_Genes"  # 【提示】确保这里是您的 txt 文件所在路径
all_genes = set()
tissue_genes = {}

for file in os.listdir(folder_path):
    if file.endswith('.txt') and "Knee_Genes" in file:
        tissue_name = re.sub(r'_Knee_Genes_\d+\.txt', '', file).replace('_', ' ')
        with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
            genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
            tissue_genes[tissue_name] = genes
            all_genes.update(genes)

# ==========================================
# 4. 获取归类并计算比例
# ==========================================
global_gene_dict = get_objective_classifications(list(all_genes))

# 更新分类顺序 (核糖体大类放在首位)
categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]

results = []
for tissue, genes in tissue_genes.items():
    if not genes: continue
    counts = {cat: 0 for cat in categories}
    for g in genes:
        cat = global_gene_dict.get(g, 'Other / Unclassified')
        counts[cat] += 1

    total = sum(counts.values())
    perc = {cat: (counts[cat]/total)*100 for cat in categories}
    perc['Tissue'] = tissue
    results.append(perc)

df = pd.DataFrame(results)
if df.empty:
    print("未生成任何数据，可能是当前目录下没有找到符合 '_Knee_Genes_' 命名的 txt 文件！")
else:
    df.set_index('Tissue', inplace=True)
    # 强制图表排序：按照核糖体类目从高到低排列
    df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)
    
    # 【非常重要】导出详细结果表，用于应对审稿人质疑
    df.to_csv(os.path.join(folder_path, "Gene_Classification_Data.csv"))
    pd.DataFrame.from_dict(global_gene_dict, orient='index', columns=['Category']).to_csv(os.path.join(folder_path, "Gene_to_Category_Map.csv"))

    # ==========================================
    # 5. 绘制主刊级堆叠柱状图
    # ==========================================
    fig, ax = plt.subplots(figsize=(16, 8))

    colors = [
        '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
        '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
    ]

    ax.yaxis.grid(True, color='#D3D3D3', linestyle='--', linewidth=0.7, zorder=0)

    df[categories].plot(kind='bar', stacked=True, ax=ax, color=colors, 
                        width=0.8, edgecolor='white', linewidth=0.7, zorder=3)

    ax.set_ylim(0, 100)
    ax.set_ylabel('Percentage of Genes (%)', fontsize=14, fontweight='bold', labelpad=10)
    ax.set_xlabel('', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=12)
    plt.yticks(fontsize=12)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=11, 
                       frameon=False, title='Gene Functional Class', title_fontsize=13, labelspacing=0.8)
    legend.get_title().set_fontweight('bold')

    plt.tight_layout()
    plt.savefig(os.path.join(folder_path, "Aging_Genes_Proportions_Final.pdf"))
    plt.savefig(os.path.join(folder_path, "Aging_Genes_Proportions_Final.png"), dpi=300)
    print("✅ 深度 GO 词元正则归类完毕！图表及分类字典映射表 (CSV) 已生成。")
    plt.show()

人类-gene-ratio

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================
# 1. 主刊级图表全局设置
# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
mpl.rcParams['axes.linewidth'] = 1.2

# 新增：正则匹配辅助函数，避免无边界字符串的错误匹配 (如 'camp' 错误匹配到 'hippocampus')
def match_keywords(text, keywords):
    for k in keywords:
        # \b 表示单词边界，re.escape 确保特殊字符安全解析
        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================
# 2. 自动化获取全球数据库的基因客观分类 (人类特化 + 严格正则版)
# ==========================================
def get_human_gene_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个人类基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    
    # 获取包含 GO 注释的完整信息，注意这里 species 已经设定为 'human'
    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='human')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()

            # 提取 GO 注释并拼接
            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            
            # 构建综合检索文本
            search_text = desc + " " + go_terms + " " + gene_type

            # --- 严格正则分类判断引擎 ---
            
            # 1. 过滤非编码RNA与未定义基因
            if 'rna' in gene_type or 'pseudo' in gene_type or symbol.startswith(('MIR', 'LINC', 'LOC')):
                category = 'ncRNA & Uncharacterized'
            
            # 2. 【最高优先级】：核糖体、RNA加工与翻译
            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                
            # 3. 细胞骨架与 ECM
            elif symbol.startswith(('COL', 'MMP', 'ACT', 'MYO', 'KRT')) or \
                 match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                
            # 4. 免疫与防御
            elif symbol.startswith(('HLA', 'CXCL', 'CCL', 'IL', 'CD', 'IG')) or \
                 match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                
            # 5. 细胞周期与凋亡
            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                
            # 6. 蛋白折叠与降解
            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                
            # 7. 表观遗传与染色质
            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                
            # 8. 转录调控
            elif symbol.startswith(('ZNF', 'STAT', 'SMAD', 'HOX', 'FOX')) or \
                 match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                
            # 9. 发育与分化
            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                
            # 10. 囊泡运输与细胞膜
            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                
            # 11. 信号传导与受体
            elif 'R' in symbol[-2:] or match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                
            # 12. 代谢与转运
            elif symbol.startswith(('SLC', 'ATP', 'NDUF')) or \
                 match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                
            # 13. 其他
            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================
# 3. 读取本地文件并精准提取组织名
# ==========================================
# ⚠️ 注意：请将这里的路径替换为您存放基因txt文件的真实路径！
folder_path = "/mnt/e/2-8.3-shanda/1-feature/1-human-guaidian-choose-gene/Gene_Lists"

all_genes = set()
tissue_genes = {}

# 如果您本地文件夹不在这个路径下，为了防止代码直接崩溃，这里做一个防护
if not os.path.exists(folder_path):
    print(f"❌ 找不到路径: {folder_path}，请修改代码中的 folder_path！")
else:
    for file in os.listdir(folder_path):
        if file.endswith('.txt'):
            # --- 🌟 专门针对 type_7_large_intestine_Knee_29_Genes.txt 的解析规则 🌟 ---
            name_clean = file
            # 1. 砍掉前面的 "type_数字_" (如 type_7_)
            name_clean = re.sub(r'^type_\d+_', '', name_clean, flags=re.IGNORECASE)
            # 2. 砍掉后面的 "_Knee_数字_Genes.txt" 或类似的后缀
            name_clean = re.sub(r'_Knee_\d+_Genes\.txt$', '', name_clean, flags=re.IGNORECASE)
            # 兜底去除 .txt 和 Genes 等词
            name_clean = re.sub(r'_(Knee|Genes|Gene).*$', '', name_clean, flags=re.IGNORECASE)
            name_clean = name_clean.replace('.txt', '')

            # 3. 将 "large_intestine" 替换为 "Large Intestine" (美观的标题大小写)
            tissue_name = name_clean.replace('_', ' ').title()
            # -------------------------------------------------------------------------

            file_path = os.path.join(folder_path, file)
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                # 提取干净的基因名
                genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
                tissue_genes[tissue_name] = genes
                all_genes.update(genes)

    human_gene_dict = get_human_gene_classifications(list(all_genes))

    # ==========================================
    # 4. 统计各组织的比例结构
    # ==========================================
    categories = [
        'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
        'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
        'Transcription Regulation', 'Development & Differentiation', 
        'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
        'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
    ]

    results = []
    for tissue, genes in tissue_genes.items():
        if not genes: continue
        counts = {cat: 0 for cat in categories}
        for g in genes:
            cat = human_gene_dict.get(g, 'Other / Unclassified')
            counts[cat] += 1

        total = sum(counts.values())
        perc = {cat: (counts[cat]/total)*100 for cat in categories}
        perc['Tissue'] = tissue
        results.append(perc)

    df = pd.DataFrame(results)
    
    if not df.empty:
        df.set_index('Tissue', inplace=True)
        # 强制图表排序：按照核糖体类目从高到低排列
        df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)

        # ==========================================
        # 5. 绘制主刊级基因类型比例图
        # ==========================================
        fig, ax = plt.subplots(figsize=(16, 8))

        # 扩充配色以适应 13 个类别
        colors = [
            '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
            '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
        ]
        ax.yaxis.grid(True, color='#D3D3D3', linestyle='--', linewidth=0.7, zorder=0)

        df[categories].plot(kind='bar', stacked=True, ax=ax,
                            color=colors, width=0.8,
                            edgecolor='white', linewidth=0.7, zorder=3)

        ax.set_ylim(0, 100)
        ax.set_ylabel('Percentage of Genes (%)', fontsize=14, fontweight='bold', labelpad=10)
        ax.set_xlabel('', fontsize=14)
        plt.xticks(rotation=45, ha='right', fontsize=12)
        plt.yticks(fontsize=12)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5),
                           fontsize=11, frameon=False, title='Gene Functional Class',
                           title_fontsize=13, labelspacing=0.8)
        legend.get_title().set_fontweight('bold')

        plt.tight_layout()
        
        # 导出为 PDF (矢量图格式，主刊级必备)
        output_pdf = os.path.join(folder_path, "1-figure-Human_Gene_Classification_Proportions.pdf")
        # 额外导出一份高分辨率的 PNG，方便快速查看
        output_png = os.path.join(folder_path, "1-figure-Human_Gene_Classification_Proportions.png")

        # bbox_inches='tight' 防止右侧的图例被画面截断
        plt.savefig(output_pdf, bbox_inches='tight') 
        plt.savefig(output_png, dpi=300, bbox_inches='tight')

        print(f"✅ 图表已成功生成！")
        print(f"📄 PDF 矢量图已保存至: {output_pdf}")
        print(f"🖼️ PNG 预览图已保存至: {output_png}")
        plt.show()
    else:
        print("没有可用的数据生成图表，请检查文本文件。")

词云图

In [ ]:
import os
import pandas as pd
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from PIL import Image

# ==================== 1. 配置路径 ====================
INPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Final_Segmented_Genes"
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Summary_Results"

# ==================== 2. 主刊级别视觉配置 ====================
# 推荐配色：'plasma' (紫-红-黄), 'magma' (黑-紫-橘), 'inferno' (黑-红-黄)
# 这些配色在红绿色盲眼中依然具有极高的辨识度。
COLOR_MAP = 'plasma'
MAX_WORDS = 100 # 主刊图表通常追求“少而精”，突出核心 Marker
DPI = 300

# ==================== 3. 核心处理逻辑 ====================
os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_oval_mask(width=1600, height=900):
    """生成椭圆形的掩模阵列"""
    x, y = np.ogrid[:height, :width]
    center_x, center_y = width / 2, height / 2
    # 椭圆方程: (x-h)^2/a^2 + (y-k)^2/b^2 <= 1
    # a, b 控制水平和垂直半径
    mask = ((x - center_y) ** 2 / (height * 0.45) ** 2 +
            (y - center_x) ** 2 / (width * 0.45) ** 2) > 1
    return 255 * mask.astype(int)

if not os.path.exists(INPUT_DIR):
    print(f"错误: 找不到目录 {INPUT_DIR}")
else:
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]
    all_genes = []

    for filename in files:
        file_path = os.path.join(INPUT_DIR, filename)
        with open(file_path, 'r') as f:
            all_genes.extend([line.strip().upper() for line in f if line.strip()])

    if not all_genes:
        print("未找到基因数据。")
    else:
        # 统计频率
        gene_freq = dict(Counter(all_genes))

        # --- 生成椭圆词云 ---
        print(f"正在生成主刊级椭圆词云 (Color-safe: {COLOR_MAP})...")

        oval_mask = create_oval_mask(2000, 1200) # 提高掩模分辨率

        wc = WordCloud(
            width=2000, height=1200,
            background_color='white',
            mask=oval_mask,
            colormap=COLOR_MAP,
            max_words=MAX_WORDS,
            min_font_size=10,
            max_font_size=250,      # 拉大字号差异，增强视觉等级
            random_state=42,
            prefer_horizontal=0.9,  # 90% 横向，保持生信论文的整洁感
            relative_scaling=0.5,   # 频率与大小的平衡系数
            contour_width=1,        # 增加极细轮廓
            contour_color='#eeeeee'
        )

        wc.generate_from_frequencies(gene_freq)

        # 绘图
        plt.figure(figsize=(20, 12))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")

        # --- 保存出版级文件 ---
        out_base = os.path.join(OUTPUT_DIR, "Aging_Gene_Cloud_Oval")
        # PDF 格式：投稿首选，矢量不失真
        plt.savefig(f"{out_base}.pdf", dpi=DPI, format='pdf', bbox_inches='tight', pad_inches=0.1)
        # PNG 格式：高分预览
        plt.savefig(f"{out_base}.png", dpi=DPI, format='png', bbox_inches='tight', pad_inches=0.1)

        plt.show()

        print("\n" + "="*40)
        print(f"✅ 处理完成！")
        print(f"1. 椭圆词云 (PDF/PNG) 已保存至: {OUTPUT_DIR}")
        print(f"2. 使用色图: {COLOR_MAP} (红绿色盲友好)")
        print("="*40)

小鼠-首字母为大写，其余小写-词云图-end

In [ ]:
import os
import pandas as pd
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import platform

# ==================== 1. 配置路径 ====================
# ==================== 1. 配置路径 ====================
INPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Final_Segmented_Genes"
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/2-ciyuntu"

# --- 核心修复：智能识别 WSL 环境并读取 Arial 字体 ---
system = platform.system()
if system == "Windows":
    FONT_PATH = "C:/Windows/Fonts/arial.ttf"
elif system == "Darwin": # macOS
    FONT_PATH = "/Library/Fonts/Arial.ttf"
else: # Linux / WSL
    # 优先尝试在 WSL 中直接访问 Windows 宿主机的字体
    wsl_font_path = "/mnt/c/Windows/Fonts/arial.ttf"
    linux_native_path = "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"
    
    if os.path.exists(wsl_font_path):
        FONT_PATH = wsl_font_path
    elif os.path.exists(linux_native_path):
        FONT_PATH = linux_native_path
    else:
        # 终极兜底：如果你不在 WSL 且没装字体，请把 arial.ttf 拷到代码运行的当前目录
        print("⚠️ 未能在系统路径中找到 Arial 字体。正在尝试使用当前目录下的 arial.ttf...")
        FONT_PATH = "arial.ttf" 

# ==================== 2. 主刊级别视觉配置 ====================
COLOR_MAP = 'plasma'
MAX_WORDS = 100 
DPI = 300

# ==================== 3. 核心处理逻辑 ====================
os.makedirs(OUTPUT_DIR, exist_ok=True)

def create_oval_mask(width=2000, height=1200):
    """生成椭圆形的掩模阵列"""
    x, y = np.ogrid[:height, :width]
    center_x, center_y = width / 2, height / 2
    # 椭圆方程
    mask = ((x - center_y) ** 2 / (height * 0.45) ** 2 +
            (y - center_x) ** 2 / (width * 0.45) ** 2) > 1
    return 255 * mask.astype(int)

if not os.path.exists(INPUT_DIR):
    print(f"错误: 找不到目录 {INPUT_DIR}")
else:
    files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.txt')]
    all_genes = []

    for filename in files:
        file_path = os.path.join(INPUT_DIR, filename)
        with open(file_path, 'r') as f:
            # 保证小鼠基因命名规范 (首字母大写，其余小写)
            all_genes.extend([line.strip().capitalize() for line in f if line.strip()])

    if not all_genes:
        print("未找到基因数据。")
    else:
        gene_freq = dict(Counter(all_genes))

        print(f"正在生成主刊级椭圆词云 (Color-safe: {COLOR_MAP})...")

        oval_mask = create_oval_mask(2000, 1200) 

        try:
            wc = WordCloud(
                font_path=FONT_PATH,    # 使用修复后的 Arial 路径
                width=2000, height=1200,
                background_color='white',
                mask=oval_mask,
                colormap=COLOR_MAP,
                max_words=MAX_WORDS,
                min_font_size=12,
                max_font_size=250,      
                random_state=42,
                prefer_horizontal=1.0,  
                relative_scaling=0.5,   
                contour_width=0,        
            )

            wc.generate_from_frequencies(gene_freq)

            # --- 绘图展示 (双栏物理尺寸 7英寸 x 4.2英寸) ---
            plt.figure(figsize=(7, 4.2))
            plt.imshow(wc, interpolation="bilinear")
            plt.axis("off")

            # --- 保存出版级文件 ---
            out_base = os.path.join(OUTPUT_DIR, "Aging_Gene_Cloud_Oval")
            
            # 1. 保存高分 PNG 预览
            plt.savefig(f"{out_base}.png", dpi=DPI, format='png', bbox_inches='tight', pad_inches=0.1)
            
            # 2. 保存包裹图片的 PDF
            plt.savefig(f"{out_base}.pdf", dpi=DPI, format='pdf', bbox_inches='tight', pad_inches=0.1)
            
            # 3. 保存纯文本矢量 SVG (强烈推荐用于 Adobe Illustrator 排版)
            svg_text = wc.to_svg(embed_font=True)
            with open(f"{out_base}.svg", "w", encoding="utf-8") as f:
                f.write(svg_text)

            plt.show()

            print("\n" + "="*40)
            print(f"✅ 处理完成！")
            print(f"1. 基因已规范化为小鼠格式 (如 Sox2)")
            print(f"2. 词云已保存为 PNG, PDF 和 纯矢量 SVG 格式")
            print(f"3. 字体成功调用: {FONT_PATH}")
            print("="*40)
            
        except OSError as e:
            print("\n❌ 严重错误: 依然无法加载 Arial 字体。")
            print("请尝试以下终极解决方案：")
            print("1. 在 Windows 中打开 C:\\Windows\\Fonts")
            print("2. 复制 arial.ttf 文件到你的 Linux 代码目录 (/mnt/e/2-8.3-shanda/1-feature/)")
            print("3. 再次运行本代码")

1-mouse-gene-ratio-end

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import re
import mygene

# ==========================================
# 1. 图表全局设置 (主刊级极简细线风格)
# ==========================================
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
# 主刊要求极细的坐标轴线条，推荐 0.5pt
mpl.rcParams['axes.linewidth'] = 0.5 

# 新增：正则匹配辅助函数，避免无边界字符串的错误匹配
def match_keywords(text, keywords):
    for k in keywords:
        # \b 表示单词边界，re.escape 确保特殊字符安全解析
        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

# ==========================================
# 2. 从联网数据库获取基因客观分类 
# ==========================================
def get_objective_classifications(gene_list):
    if not gene_list:
        print("警告：未读取到任何基因，请检查 txt 文件路径。")
        return {}

    print(f"正在深度联网查询 {len(gene_list)} 个小鼠基因的名称与 GO 功能库，请稍候...")
    mg = mygene.MyGeneInfo()
    
    # 获取包含 GO 注释的完整信息
    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='mouse')

    gene_info_dict = {}
    for res in results:
        query_gene = res.get('query')
        if query_gene and query_gene not in gene_info_dict:
            gene_info_dict[query_gene] = res

    classification_dict = {}
    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()

            # 提取 GO 注释并拼接
            go_terms = ""
            if 'go' in row:
                go_data = row['go']
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in go_data:
                        items = go_data[sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items:
                                go_terms += str(item.get('term', '')).lower() + " "
            
            search_text = desc + " " + go_terms + " " + gene_type

            # --- 严格正则分类判断引擎 ---
            
            # 1. 过滤未定义基因与假基因
            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            
            # 2. 【最高优先级】：核糖体、RNA加工与翻译
            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or \
                 match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
                
            # 3. 细胞骨架与 ECM
            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
                
            # 4. 免疫与防御
            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
                
            # 5. 细胞周期与凋亡
            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
                
            # 6. 蛋白折叠与降解
            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
                
            # 7. 表观遗传与染色质
            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
                
            # 8. 转录调控
            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
                
            # 9. 发育与分化
            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
                
            # 10. 囊泡运输与细胞膜
            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
                
            # 11. 信号传导与受体
            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
                
            # 12. 代谢与转运
            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
                
            # 13. 其他
            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================
# 3. 路径分离配置与本地文件读取
# ==========================================
# 您的 WSL/Linux 路径配置
INPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Final_Segmented_Genes" 
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/3-ratio-mouse-gene-tissue"

os.makedirs(OUTPUT_DIR, exist_ok=True)

all_genes = set()
tissue_genes = {}

for file in os.listdir(INPUT_DIR):
    if file.endswith('.txt') and "Knee_Genes" in file:
        tissue_name = re.sub(r'_Knee_Genes_\d+\.txt', '', file).replace('_', ' ')
        with open(os.path.join(INPUT_DIR, file), 'r', encoding='utf-8') as f:
            genes = [line.strip().split()[-1].upper() for line in f.readlines() if line.strip()]
            tissue_genes[tissue_name] = genes
            all_genes.update(genes)

# ==========================================
# 4. 获取归类并计算比例
# ==========================================
global_gene_dict = get_objective_classifications(list(all_genes))

categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]

results = []
for tissue, genes in tissue_genes.items():
    if not genes: continue
    counts = {cat: 0 for cat in categories}
    for g in genes:
        cat = global_gene_dict.get(g, 'Other / Unclassified')
        counts[cat] += 1

    total = sum(counts.values())
    perc = {cat: (counts[cat]/total)*100 for cat in categories}
    perc['Tissue'] = tissue
    results.append(perc)

df = pd.DataFrame(results)
if df.empty:
    print(f"未生成任何数据，可能是 {INPUT_DIR} 目录下没有找到符合 '_Knee_Genes_' 命名的 txt 文件！")
else:
    df.set_index('Tissue', inplace=True)
    df = df.sort_values(by='Ribosome, RNA & Translation', ascending=False)
    
    df.to_csv(os.path.join(OUTPUT_DIR, "Gene_Classification_Data.csv"))
    pd.DataFrame.from_dict(global_gene_dict, orient='index', columns=['Category']).to_csv(os.path.join(OUTPUT_DIR, "Gene_to_Category_Map.csv"))

    # ==========================================
    # 5. 绘制主刊级堆叠柱状图 (终极抛光版)
    # ==========================================
    # --- 精准锁定物理尺寸: 126 mm x 60 mm ---
    width_in = 126 / 25.4   
    height_in = 60 / 25.4   
    fig, ax = plt.subplots(figsize=(width_in, height_in))

    # --- 坚持使用您指定的 NPG 经典原版配色 ---
    colors = [
        '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
        '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
    ]

    ax.yaxis.grid(True, color='#E6E6E6', linestyle='-', linewidth=0.5, zorder=0)

    # 堆叠图绘制
    df[categories].plot(kind='bar', stacked=True, ax=ax, color=colors, 
                        width=0.8, edgecolor='white', linewidth=0.5, zorder=3)

    ax.set_ylim(0, 100)
    
    # 细化 Y 轴标题设置
    ax.set_ylabel('Percentage of Genes (%)', fontsize=7, fontweight='normal', labelpad=2)
    ax.set_xlabel('', fontsize=7)
    
    plt.xticks(rotation=45, ha='right', fontsize=6)
    plt.yticks(fontsize=6)
    
    # --- 核心修改：将 X 轴和 Y 轴的 pad 分开控制 ---
    # Y 轴保留 pad=2 防止数字挤压，X 轴 pad 设为 0 让文字紧贴刻度线
    ax.tick_params(axis='y', which='major', width=0.5, length=2.5, pad=2)
    ax.tick_params(axis='x', which='major', width=0.5, length=2.5, pad=0)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # 图例设置保持紧凑
    legend = ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=5.5, 
                       frameon=False, title='Gene Functional Class', title_fontsize=6.5, 
                       labelspacing=0.6, handlelength=1.0, handleheight=0.6)
    legend.get_title().set_fontweight('bold')

    plt.tight_layout()
    
    output_pdf = os.path.join(OUTPUT_DIR, "Aging_Genes_Proportions_Final.pdf")
    output_png = os.path.join(OUTPUT_DIR, "Aging_Genes_Proportions_Final.png")
    # 新增 SVG 导出
    output_svg = os.path.join(OUTPUT_DIR, "Aging_Genes_Proportions_Final.svg")
    
    plt.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03, facecolor='white')
    plt.savefig(output_png, dpi=300, format='png', bbox_inches='tight', pad_inches=0.03, facecolor='white')
    # 保存 SVG
    plt.savefig(output_svg, format='svg', bbox_inches='tight', pad_inches=0.03, facecolor='white')

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import Counter
import platform
import matplotlib.font_manager as font_manager
import re
import mygene

# ==========================================
# 1. 路径与环境配置
# ==========================================
INPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Final_Segmented_Genes"
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/2-ciyuntu"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 主刊级极简细线风格与真字体配置 ---
system = platform.system()
if system == "Windows":
    font_path = "C:/Windows/Fonts/arial.ttf"
elif system == "Darwin": 
    font_path = "/Library/Fonts/Arial.ttf"
else: 
    font_path = "/mnt/c/Windows/Fonts/arial.ttf"
    if not os.path.exists(font_path):
        font_path = "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf"

if os.path.exists(font_path):
    font_manager.fontManager.addfont(font_path)
    mpl.rcParams['font.family'] = 'Arial'
else:
    mpl.rcParams['font.family'] = 'sans-serif'
    mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['axes.linewidth'] = 0.5 

# ==========================================
# 2. 数据库分类引擎 (与之前热图逻辑完全一致)
# ==========================================
def match_keywords(text, keywords):
    for k in keywords:
        if re.search(r'\b' + re.escape(k) + r'\b', text):
            return True
    return False

def get_objective_classifications(gene_list):
    if not gene_list: return {}
    print(f"正在深度联网查询 {len(gene_list)} 个核心基因的 GO 功能库...")
    mg = mygene.MyGeneInfo()
    results = mg.querymany(gene_list, scopes='symbol', fields='name,type_of_gene,go', species='mouse')

    gene_info_dict = {res.get('query'): res for res in results if res.get('query')}
    classification_dict = {}

    for gene in gene_list:
        try:
            row = gene_info_dict.get(gene, {})
            symbol = str(gene).upper()
            desc = str(row.get('name', '')).lower()
            gene_type = str(row.get('type_of_gene', '')).lower()

            go_terms = ""
            if 'go' in row:
                for sub_ont in ['BP', 'MF', 'CC']:  
                    if sub_ont in row['go']:
                        items = row['go'][sub_ont]
                        if isinstance(items, dict): items = [items]
                        if isinstance(items, list):
                            for item in items: go_terms += str(item.get('term', '')).lower() + " "
            
            search_text = desc + " " + go_terms + " " + gene_type

            # --- 分类判断引擎 ---
            if 'rna' in gene_type or 'pseudo' in gene_type or re.match(r'^GM\d+$', symbol) or symbol.endswith('RIK') or re.match(r'^BC\d+$', symbol):
                category = 'ncRNA & Uncharacterized'
            elif symbol.startswith(('RPL', 'RPS', 'MRPL', 'MRPS', 'EIF', 'EEF', 'SNRP', 'HNRNP', 'DDX')) or match_keywords(search_text, ['ribosome', 'ribosomal', 'translation', 'trna', 'rrna', 'mrna', 'elongation factor', 'initiation factor', 'rna binding', 'spliceosome', 'splicing', 'rna processing', 'ribonucleoprotein']):
                category = 'Ribosome, RNA & Translation'
            elif match_keywords(search_text, ['collagen', 'matrix', 'actin', 'myosin', 'adhesion', 'elastin', 'cytoskeleton', 'keratin', 'tubulin', 'vimentin', 'integrin', 'laminin', 'fibronectin', 'cadherin', 'microtubule', 'extracellular']):
                category = 'Cytoskeleton & ECM'
            elif match_keywords(search_text, ['immune', 'chemokine', 'interleukin', 'cd antigen', 'histocompatibility', 'complement', 'lysozyme', 's100', 'interferon', 'macrophage', 't cell', 'b cell', 'antigen', 'toll-like', 'defensin', 'leukocyte', 'inflammatory', 'mhc', 'phagocytosis']):
                category = 'Immunity & Defense'
            elif match_keywords(search_text, ['cell cycle', 'apoptosis', 'cyclin', 'caspase', 'cell division', 'mitosis', 'meiosis', 'senescence', 'death', 'p53', 'proliferation']):
                category = 'Cell Cycle & Apoptosis'
            elif match_keywords(search_text, ['ubiquitin', 'proteasome', 'chaperone', 'heat shock', 'autophagy', 'peptidase', 'protease', 'degradation', 'protein folding']):
                category = 'Protein Folding & Degradation'
            elif match_keywords(search_text, ['histone', 'chromatin', 'methyltransferase', 'acetyltransferase', 'epigenetic', 'nucleosome', 'hdac', 'dna methylation']):
                category = 'Epigenetics & Chromatin'
            elif match_keywords(search_text, ['transcription', 'polymerase', 'zinc finger', 'box', 'promoter']):
                category = 'Transcription Regulation'
            elif match_keywords(search_text, ['development', 'differentiation', 'morphogenesis', 'homeobox', 'hox', 'embryonic', 'stem cell', 'angiogenesis', 'osteogenesis', 'chondrogenesis']):
                category = 'Development & Differentiation'
            elif match_keywords(search_text, ['vesicle', 'exocytosis', 'endocytosis', 'rab', 'snare', 'clathrin', 'golgi', 'endoplasmic reticulum', 'membrane', 'peroxisome', 'lysosome', 'vacuole']):
                category = 'Vesicular Transport & Cytomembrane'
            elif match_keywords(search_text, ['kinase', 'phosphatase', 'receptor', 'signal', 'g-protein', 'ras', 'wnt', 'tgf', 'bmp', 'notch', 'hedgehog', 'hormone', 'calcium', 'camp']):
                category = 'Signaling & Receptors'
            elif match_keywords(search_text, ['atp', 'cytochrome', 'metabolism', 'dehydrogenase', 'synthase', 'solute carrier', 'transporter', 'lipid', 'cholesterol', 'fatty acid', 'glycolysis', 'mitochondrial', 'channel', 'pump', 'oxidoreductase', 'catalase', 'transferase', 'reductase', 'catabolic']):
                category = 'Metabolism & Transport'
            else:
                category = 'Other / Unclassified'

            classification_dict[gene] = category
        except Exception:
            classification_dict[gene] = 'Other / Unclassified'

    return classification_dict

# ==========================================
# 3. 经典 NPG 色板映射
# ==========================================
categories = [
    'Ribosome, RNA & Translation', 'Cytoskeleton & ECM', 'Immunity & Defense',
    'Cell Cycle & Apoptosis', 'Protein Folding & Degradation', 'Epigenetics & Chromatin', 
    'Transcription Regulation', 'Development & Differentiation', 
    'Vesicular Transport & Cytomembrane', 'Signaling & Receptors', 
    'Metabolism & Transport', 'ncRNA & Uncharacterized', 'Other / Unclassified'
]
colors = [
    '#E64B35', '#4DBBD5', '#00A087', '#3C5488', '#F39B7F', '#8491B4', 
    '#91D1C2', '#DC0000', '#7E6148', '#B09C85', '#9467BD', '#FF9896', '#E0E0E0'
]
color_map_dict = dict(zip(categories, colors))

# ==========================================
# 4. 数据提取与统计
# ==========================================
all_genes = []
for file in os.listdir(INPUT_DIR):
    if file.endswith('.txt'):
        with open(os.path.join(INPUT_DIR, file), 'r', encoding='utf-8') as f:
            for line in f:
                gene = line.strip().upper() # 统一转大写用于精准统计和查询
                # 过滤空行和带有SOURCE的污染行
                if gene and "SOURCE" not in gene:
                    all_genes.append(gene)

if not all_genes:
    print(f"❌ 错误: 在 {INPUT_DIR} 中没有提取到任何基因数据。")
    exit()

# 统计频次并提取 Top 30
gene_counter = Counter(all_genes)
top_30 = gene_counter.most_common(30)
top_genes = [item[0] for item in top_30]

# 联网获取功能分类
classifications = get_objective_classifications(top_genes)

# 构建 DataFrame
df = pd.DataFrame(top_30, columns=['Gene', 'Frequency'])
df['Category'] = df['Gene'].map(lambda x: classifications.get(x, 'Other / Unclassified'))
df['Color'] = df['Category'].map(color_map_dict)

# 转换为小鼠基因命名规范 (首字母大写)，这是发表时的必须要求
df['Gene'] = df['Gene'].str.capitalize()

# 为了画图时频率最高的在最上面，我们需要倒序排列 DataFrame
df = df.sort_values(by='Frequency', ascending=True).reset_index(drop=True)

# 保存用于支撑图表的数据表 (Source Data)
df.to_csv(os.path.join(OUTPUT_DIR, "Top30_Genes_Frequency_Data.csv"), index=False)

# ==========================================
# 5. 绘制主刊级棒棒糖图 (Lollipop Plot)
# ==========================================
# 精准物理尺寸：宽 126 mm，高 90 mm (为 30 个基因留足空间)
width_in = 126 / 25.4
height_in = 90 / 25.4
fig, ax = plt.subplots(figsize=(width_in, height_in))

# 极简垂直网格线 (仅 X 轴)
ax.xaxis.grid(True, color='#E6E6E6', linestyle='-', linewidth=0.5, zorder=0)

# 绘制“糖棍” (细线)
ax.hlines(y=df.index, xmin=0, xmax=df['Frequency'], color=df['Color'], linewidth=1.0, zorder=3)

# 绘制“糖果” (带极细白色描边的圆点，增加精致感)
ax.scatter(df['Frequency'], df.index, color=df['Color'], s=25, edgecolor='white', linewidth=0.5, zorder=4)

# 坐标轴与刻度设置
ax.set_yticks(df.index)
# 🌟绝杀细节：基因名字全部使用斜体 (fontstyle='italic')
ax.set_yticklabels(df['Gene'], fontsize=6, fontstyle='italic')

ax.set_xlabel('Gene Occurrence Frequency', fontsize=7, fontweight='bold', labelpad=4)
ax.tick_params(axis='both', which='major', width=0.5, length=2.5, labelsize=6)

# 隐藏边框，保留极简风格
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)

# --- 智能图例生成 ---
# 仅提取出现在 Top 30 中的功能分类，且按照原分类列表的顺序排序
present_categories = df['Category'].unique()
legend_cats = [c for c in categories if c in present_categories]

# 自定义图例点
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map_dict[c], markersize=4.5) 
    for c in legend_cats
]

legend = ax.legend(legend_elements, legend_cats, loc='center left', bbox_to_anchor=(1.02, 0.5), 
                   fontsize=5.5, frameon=False, title='Gene Functional Class', title_fontsize=6.5, 
                   labelspacing=0.8, handletextpad=0.2)
legend.get_title().set_fontweight('bold')

plt.tight_layout()

# 保存主刊级矢量图和预览图
output_pdf = os.path.join(OUTPUT_DIR, "Fig_Top30_Genes_Lollipop.pdf")
output_png = os.path.join(OUTPUT_DIR, "Fig_Top30_Genes_Lollipop.png")

plt.savefig(output_pdf, format='pdf', bbox_inches='tight', pad_inches=0.03)
plt.savefig(output_png, dpi=300, format='png', bbox_inches='tight', pad_inches=0.03)

print("\n" + "="*50)
print(f"✅ 主刊级棒棒糖图 (Lollipop Plot) 生成完毕！")
print(f"📏 物理尺寸: 126 mm x 90 mm")
print(f"📊 已提取 Top 30 基因，并完成斜体 (Italic) 规范化")
print(f"🎨 色板已完美匹配 13 种 NPG 经典功能分类")
print(f"📂 图表与数据 (CSV) 保存至: \n   {OUTPUT_DIR}")
print("="*50)
plt.show()